# Man-Made Feature Data Preparation

This notebook extracts and cleans relevant OpenStreetMap `man_made` features and maps them into broader categories that provide contextual evidence for industrial and infrastructure-related building functions.

**Input**
- Raw OpenStreetMap data for Germany.

**Output**
- Cleaned and categorized man-made polygon features for subsequent feature engineering and industrial-site analysis.

In [1]:
import geopandas as gpd
import pyrosm
import os

ROOT_DIR = '/fast/home/o-olajuyigbe/osm_project'
DATA_DIR = os.path.join(ROOT_DIR, 'data')
PBF_FILE = os.path.join(DATA_DIR, 'raw', 'germany-latest.osm.pbf')

osm = pyrosm.OSM(PBF_FILE)

In [3]:
# ── manmade ──────────────────────────────────────────────────────────
print("Extracting manmade features...")

manmade = osm.get_data_by_custom_criteria(
    custom_filter={'man_made': True},
    filter_type='keep', keep_nodes=False, keep_ways=True, keep_relations=True
)

print(f"Raw manmade rows  : {len(manmade):,}")
print(manmade.geometry.geom_type.value_counts())
print(manmade.columns.tolist())

Extracting manmade features...
Raw manmade rows  : 427,137
Polygon            268754
MultiLineString     90932
LineString          66864
MultiPolygon          587
Name: count, dtype: int64
['man_made', 'id', 'timestamp', 'version', 'tags', 'osm_type', 'geometry', 'changeset']


In [6]:
gdf_manmade = manmade.copy()    

In [5]:
manmade.to_parquet(os.path.join(DATA_DIR, 'raw', 'germany_manmade.parquet'), index=False)

In [2]:
gdf_manmade = gpd.read_parquet(os.path.join(DATA_DIR, 'raw', 'germany_manmade.parquet'))

In [36]:
gdf_manmade.man_made.value_counts().head(20)

man_made
embankment           58344
storage_tank         46418
pier                 44279
bridge               42999
bunker_silo          41688
silo                 26075
pipeline             20344
tower                16416
cutline              15798
works                15041
groyne               13264
wastewater_plant     10128
clearcut              9857
street_cabinet        8279
dyke                  6912
reservoir_covered     6005
water_works           5183
gantry                4702
goods_conveyor        2855
stele                 2643
Name: count, dtype: int64

In [37]:
# check geometr type
gdf_manmade.geometry.geom_type.value_counts()

Polygon            268754
MultiLineString     90932
LineString          66864
MultiPolygon          587
Name: count, dtype: int64

In [38]:
import sys
SCRIPTS   = os.path.join(ROOT_DIR, 'scripts')

if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)

import importlib
import ind_grouping_map

importlib.reload(ind_grouping_map)

from ind_grouping_map import IND_MAP


Counter({'food_processing': 37, 'engineering': 33, 'water_utilities': 25, 'construction_materials': 22, 'grid_infrastructure': 21, 'chemical': 17, 'logistics': 17, 'wood_processing': 14, 'light_manufacturing': 13, 'energy_generation': 11, 'mining_quarry': 11, 'waste_management': 11, 'metal_industry': 10, 'pharmaceutical': 9, 'manufacturing': 8, 'petroleum': 8, 'hightech_manufacturing': 8, 'storage': 6, 'warehouse': 4, 'biogas': 3, 'heating_supply': 3, 'cold_storage': 2, 'silo': 2, 'grid_infrasturucture': 1})


In [ ]:
# Keep only polygon geometries (same as buildings)
VALID_WATERWAY_GEOMS = {'Polygon', 'MultiPolygon'}
gdf_manmade = gdf_manmade[
    gdf_manmade.geometry.geom_type.isin(VALID_WATERWAY_GEOMS)
].copy()

print(f"After geometry filter: {len(gdf_manmade):,}")

# Drop all osm_type = 'relation' 
gdf_manmade = gdf_manmade[gdf_manmade['osm_type'] != 'relation'].copy()

print(f"After relation drop: {len(gdf_manmade):,}")

# ── Category mapping ───────────────────────────────────────────────────

gdf_manmade['manmade_category'] = gdf_manmade['man_made'].map(IND_MAP)

# Drop unmapped
gdf_manmade = gdf_manmade[gdf_manmade['manmade_category'].notna()].copy()

print(gdf_manmade['manmade_category'].value_counts())
print(f"Final manmade rows: {len(gdf_manmade):,}")

After geometry filter: 269,341
manmade_category
water_utilities           68116
silo                      67696
manufacturing             14816
grid_infrastructure        1176
mining_quarry               290
hightech_manufacturing      192
chemical                    129
storage                      96
biogas                       75
petroleum                    74
waste_management             70
logistics                    53
construction_materials       15
engineering                  11
wood_processing               7
energy_generation             7
heating_supply                5
warehouse                     2
Name: count, dtype: int64
Final manmade rows: 152,830


In [40]:
gdf_manmade[gdf_manmade['id']==38693275710]

,man_made,id,timestamp,version,tags,osm_type,geometry,changeset,manmade_category


In [41]:
# Keep only the columns needed for feature engineering
manmade_slim = gdf_manmade[['id', 'geometry', 'manmade_category', 'man_made']].copy()

# Fix invalid geometries
invalid = ~manmade_slim.geometry.is_valid
if invalid.any():
    manmade_slim.loc[invalid, 'geometry'] = manmade_slim.loc[invalid, 'geometry'].buffer(0)
    manmade_slim = manmade_slim[manmade_slim.geometry.is_valid].copy()
    print(f"Fixed {invalid.sum()} invalid geometries")

manmade_slim.to_parquet(
    os.path.join(DATA_DIR, 'processed', 'germany_manmade_mapped.parquet'),
    index=False
)
print(f"Saved germany_manmade_mapped.parquet — {len(manmade_slim):,} rows")
print(f"CRS: {manmade_slim.crs}")

Fixed 35 invalid geometries
Saved germany_manmade_mapped.parquet — 152,830 rows
CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", 